# CISC440 – Homework 9  
# Misère Nim: From Search to Learning  
## Monte Carlo Simulation and Reinforcement Learning

Name: Nikhil Jangir and Rudy Vergara

## Overview

In Homework 6, you solved **Misère Nim** using adversarial search methods such as Minimax, Alpha-Beta Pruning, and Expectimax. Those methods assume the agent reasons using explicit search trees.

In this homework, you will revisit the same game but move into a new area of AI:

- **Monte Carlo methods**: learning through simulation
- **Reinforcement Learning**: learning through repeated experience

Instead of solving the game by searching the tree exactly, your agent will estimate strong moves through simulation and learn policies through trial and error.

This homework continues the same Misère Nim environment so you can focus on new AI ideas rather than learning a new game.

## Learning Objectives

By completing this homework, you will be able to:

- Apply Monte Carlo simulation to estimate action quality
- Use repeated rollouts to choose moves
- Model learning agents using states, actions, and rewards
- Implement Q-learning for sequential decision making
- Compare search-based and learning-based approaches
- Analyze experimental performance of AI agents

## Misère Nim Rules

You are given several heaps of objects.

A move consists of:

1. Selecting one heap
2. Removing one or more objects from that heap

The player who removes the **last remaining object loses**.

This is different from normal Nim, where the player who removes the last object usually wins.

## Initial States for Testing

Use the following starting states:

```python
[1, 3, 5]
[3, 5, 7]
[1, 1, 2]
[2, 4, 6]
```

## Submission Requirements

Submit:

1. This completed Jupyter notebook
2. Any helper Python files you created
3. Clear outputs for all required experiments
4. Written explanations in the markdown cells
5. Clear comments in code

You may reuse parts of your Homework 6 Nim code, but your Monte Carlo and Q-learning work must be your own.

## Grading Breakdown: 100 Points

| Section | Points |
|---|---:|
| Layer 1 – Problem Formulation | 25 |
| Layer 2 – Monte Carlo Implementation | 30 |
| Layer 3 – Reinforcement Learning | 30 |
| Layer 4 – Experimental Comparison | 15 |

## Setup Code

In [4]:
import random
from collections import defaultdict
import pandas as pd

# Helper Functions

You may complete or modify these helper functions. These are intentionally simple so that the focus stays on Monte Carlo simulation and reinforcement learning.

In [13]:
def normalize_state(state):
    """
    Convert a state to a tuple so it can be used as a dictionary key.
    Example:
        [3, 5, 7] -> (3, 5, 7)
    """
    return tuple(state)


def is_terminal(state):
    """
    Return True if the game is over.
    In Nim, the game ends when all heaps are empty.
    """
    # The game is terminal when the sum of all heap sizes is 0
    return sum(state) == 0


def legal_actions(state):
    """
    Return all legal actions from a state.

    Action format:
        (heap_index, remove_count)
    """
    actions = []
    for heap_idx, heap_size in enumerate(state):
        # For each non-empty heap, we can remove 1 to heap_size objects
        for remove_count in range(1, heap_size + 1):
            actions.append((heap_idx, remove_count))
    return actions


def apply_action(state, action):
    """
    Apply an action and return the new state.
    Do not modify the original state.
    """
    heap_idx, remove_count = action
    new_state = list(state)  # Create a copy to avoid modifying the original
    new_state[heap_idx] -= remove_count  # Remove objects from the heap
    return new_state

# Layer 1 – Problem Formulation (25 Points)

Answer clearly in complete sentences.

## Part A – Monte Carlo Modeling (12 pts)

### Q1. State Representation (3 pts)

How will a Misère Nim board state be represented in Python?

Example:

```python
[3, 5, 7]
```

Explain what each number means.

**Your answer:**

A Misère Nim board state is represented as a list of integers, where each integer represents the number of objects in a single heap. Refrencing the example above: the first heap contains 3 objects, the second heap contains 5 objects, and the third heap contains 7 objects. The index position in the list identifies which heap it is (heap 0, heap 1, heap 2). 

### Q2. Legal Actions (3 pts)

Describe all legal actions available from:

```python
[1, 3, 5]
```

Represent actions as:

```python
(heap_index, remove_count)
```

**Your answer:**

From state `[1, 3, 5]`, the legal actions are:
- From heap 0: `(0, 1)` = remove 1 object
- From heap 1: `(1, 1)`, `(1, 2)`, `(1, 3)` = remove 1, 2, or 3 objects
- From heap 2: `(2, 1)`, `(2, 2)`, `(2, 3)`, `(2, 4)`, `(2, 5)` = remove 1, 2, 3, 4, or 5 objects

In total, there are 9 legal actions. A legal action consists of selecting one non-empty heap and removing at least one object (but not more than the heap size). Each action is represented as a tuple `(heap_index, remove_count)` where `heap_index` identifies which heap to remove from, and `remove_count` specifies how many objects to remove.

### Q3. Terminal Test (3 pts)

When does the game end?

How does Misère Nim differ from normal Nim?

**Your answer:**

In Misère Nim, the game ends when all heaps are empty which is when the state is `[0, 0, 0]`. At this point, there are no legal moves remaining.

The critical difference is in normal Nim, the player who takes the last object wins. This reversal fundamentally changes game strategy. In Misère Nim, players want to avoid being forced into a position where they must take the last remaining object. This makes certain positions that would be winning in normal Nim actually losing positions in Misère Nim, and vice versa.

### Q4. Why Monte Carlo? (3 pts)

Why might simulation be useful instead of exploring the full search tree?

**Your answer:**

Monte Carlo simulation is useful for several reasons:

1. The full game tree can be exponentially large. Sampling trajectories is often faster than exhaustively exploring all branches.

2. For many games, you don't need the exact game value; a good estimate based on random rollouts is often good enough to make strong moves.

3. As game complexity increases (more heaps, larger heap sizes), the search tree becomes intractable.

4. No need to compute minimax values or maintain complex game state trees in memory. Just simulate random games and count wins/losses.


### Q5. Define State (3 pts)

What is the RL state in this problem?

**Your answer:**

The RL state is the current configuration of heaps on the board, represented as a list of integers (e.g., `[1, 3, 5]`). The state captures all the information needed to decide what action to take next.

### Q6. Define Actions (3 pts)

What are RL actions in this problem?

**Your answer:**

RL actions are the same as legal moves in Misère Nim: tuples of the form `(heap_index, remove_count)` representing which heap to remove from and how many objects to remove. From any state, we generate all legal actions using the `legal_actions()` function. Each action transitions the current state to a new state via the `apply_action()` function.

### Q7. Reward Function (4 pts)

Define rewards for:

- Winning
- Losing
- Non-terminal moves

Explain your choice.

**Your answer:**

**Reward structure:**
- **Winning** (opponent reaches terminal state with objects left): `+1`
- **Losing** (current player forced to take last object): `-1`
- **Non-terminal moves**: `0`

**Explanation:** This simple reward structure aligns perfectly with the game objective. We want the agent to maximize rewards, so winning gives +1 and losing gives -1. Intermediate moves get 0 reward;

### Q8. Exploration vs Exploitation (3 pts)

What is the difference between exploration and exploitation?

Why are both important?

**Your answer:**

**Exploration vs Exploitation:**
- **Exploitation**: Choosing the action with the highest known Q-value. This uses what we've already learned to play well.
- **Exploration**: Occasionally choosing a random action instead. This lets us try new moves and discover better strategies we haven't found yet.

**Why both are important:**
- Both are imortant because Iif we only exploit early estimates, we may miss better strategies or get stuck at local optima. Whereas, exploration alone means we never leverage what we've learned; we'd play randomly forever and not improve. That's why balancing both of them is the key.

# Layer 2 – Monte Carlo Implementation (30 Points)

Implement a simulation-based agent.

## Part A – Random Playout Engine (10 pts)

Write:

```python
simulate_random_game(state)
```

This function should:

- Start from the given state
- Alternate players randomly
- Choose random legal moves
- Return the winner

For this homework, use player `0` as the starting player and player `1` as the other player.

Remember: in Misère Nim, the player who takes the last object loses.

In [5]:
def simulate_random_game(state, starting_player=0):
    """
    Simulate a full random Misère Nim game.

    Parameters:
        state: list of heap sizes
        starting_player: 0 or 1

    Returns:
        winner: 0 or 1
    """
    current_state = list(state)
    current_player = starting_player
    
    # Play until game is terminal (all heaps are empty)
    while not is_terminal(current_state):
        # Get all legal actions from current state
        actions = legal_actions(current_state)
        
        # Choose a random action
        action = random.choice(actions)
        
        # Apply the action to get new state
        current_state = apply_action(current_state, action)
        
        # Switch to other player
        current_player = 1 - current_player
    
    winner = current_player
    return winner

### Test your random playout engine

In [6]:
# Test state functions
print("=== Testing State Functions ===\n")

# Test is_terminal
print("Testing is_terminal():")
print(f"  is_terminal([0, 0, 0]): {is_terminal([0, 0, 0])} (should be True)")
print(f"  is_terminal([1, 3, 5]): {is_terminal([1, 3, 5])} (should be False)")
print(f"  is_terminal([0, 0, 1]): {is_terminal([0, 0, 1])} (should be False)\n")

# Test legal_actions
print("Testing legal_actions():")
state = [1, 3, 5]
actions = legal_actions(state)
print(f"  legal_actions({state}):")
print(f"  Total actions: {len(actions)}")
print(f"  Actions: {actions}\n")

# Test apply_action
print("Testing apply_action():")
state = [1, 3, 5]
action = (1, 2)  # Remove 2 from heap 1
new_state = apply_action(state, action)
print(f"  Original state: {state}")
print(f"  Action: {action} (remove {action[1]} from heap {action[0]})")
print(f"  New state: {new_state}")
print(f"  Original state unchanged: {state}\n")  # Verify original unchanged

# Test simulate_random_game
print("=== Testing Monte Carlo Functions ===\n")
print("Testing simulate_random_game():")
for i in range(5):
    winner = simulate_random_game([1, 3, 5])
    print(f"  Game {i+1}: Player {winner} wins")
print()

# Test rollout_value
print("Testing rollout_value():")
state = [1, 3, 5]
action = (2, 1)  # Remove 1 from heap 2
value = rollout_value(state, action, n_trials=50)
print(f"  State: {state}")
print(f"  Action: {action}")
print(f"  Estimated win rate: {value}\n")


=== Testing State Functions ===

Testing is_terminal():
  is_terminal([0, 0, 0]): True (should be True)
  is_terminal([1, 3, 5]): False (should be False)
  is_terminal([0, 0, 1]): False (should be False)

Testing legal_actions():
  legal_actions([1, 3, 5]):
  Total actions: 9
  Actions: [(0, 1), (1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5)]

Testing apply_action():
  Original state: [1, 3, 5]
  Action: (1, 2) (remove 2 from heap 1)
  New state: [1, 1, 5]
  Original state unchanged: [1, 3, 5]

=== Testing Monte Carlo Functions ===

Testing simulate_random_game():
  Game 1: Player 0 wins
  Game 2: Player 0 wins
  Game 3: Player 1 wins
  Game 4: Player 0 wins
  Game 5: Player 1 wins

Testing rollout_value():


NameError: name 'rollout_value' is not defined

## Part B – Action Evaluation by Rollouts (10 pts)

Write:

```python
rollout_value(state, action, n_trials=100)
```

This function should:

1. Apply the chosen action
2. Simulate random completions
3. Estimate the win rate for the player who made the original action

In [7]:
def rollout_value(state, action, n_trials=100):
    """
    Estimate the value of an action using random rollouts.

    Return:
        estimated win rate for the current player
    """
    # Apply the action to transition to next state
    next_state = apply_action(state, action)
    next_player = 1
    
    wins = 0
    # Run multiple random games from this state
    for _ in range(n_trials):
        # Simulate a random game from the new state
        winner = simulate_random_game(next_state, next_player)
        
        # Current player (player 0, who made this action) wins if winner is NOT player 1
        if winner == 0:
            wins += 1
    
    return wins / n_trials

## Part C – Monte Carlo Move Selection (10 pts)

Write:

```python
best_move_mc(state, n_trials=100)
```

Return the action with the highest estimated win probability.

In [17]:
def best_move_mc(state, n_trials=100):
    """
    Choose the best move using Monte Carlo rollouts.

    Return:
        best_action, results_table

    results_table can be a list of dictionaries or a pandas DataFrame.
    """
    actions = legal_actions(state)
    results = []
    
    best_action = None
    best_value = -1
    
    # Evaluate each legal action
    for action in actions:
        value = rollout_value(state, action, n_trials=n_trials)
        
        # Store results for reporting
        results.append({
            'action': action,
            'win_rate': value
        })
        
        # Track the best action
        if value > best_value:
            best_value = value
            best_action = action
    
    df = pd.DataFrame(results)
    df = df.sort_values('win_rate', ascending=False)  # Sort by win rate descending
    
    return best_action, df

## Required Monte Carlo Testing

Run your Monte Carlo agent on:

```python
[1, 3, 5]
[3, 5, 7]
[1, 1, 2]
```

For each state report:

- Move tested
- Estimated win rate
- Best move selected

Use a clear table.

In [16]:
test_states = [
    [1, 3, 5],
    [3, 5, 7],
    [1, 1, 2]
]

# Run best_move_mc for each state and display results
print("=== Monte Carlo Results ===\n")

for state in test_states:
    print(f"State: {state}")
    best_action, results_df = best_move_mc(state, n_trials=100)
    print(f"Best action: {best_action} with win rate: {results_df.iloc[0]['win_rate']}\n")
    print("All actions evaluated:")
    print(results_df.to_string(index=False))
    print("\n" + "="*50 + "\n")

=== Monte Carlo Results ===

State: [1, 3, 5]


NameError: name 'best_move_mc' is not defined

# Layer 3 – Reinforcement Learning (30 Points)

Implement a Q-learning agent for Misère Nim.

## Part A – Q Table Design (5 pts)

How will you store:

```python
Q[state][action]
```

Explain your structure.

**Your answer:**
We will use a nested dictionary in Python using its defaultdict. The outer Dictionary keys will be normalized game states represented as tuples. The inner dictionary for the actions keys will be legal actions we know are available from that state, also as a tuple. The Q-Values will be float numbers, representing the quality / expected outcome of taking that exact action.

## Part B – Q-learning Update Rule (10 pts)

Use:

```text
Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') − Q(s,a) ]
```

Implement:

```python
update_q(Q, s, a, r, s_next)
```

In [12]:
def update_q(Q, state, action, reward, next_state, alpha=0.5, gamma=0.9):
    """
    Apply the Q-learning update.
    """
    # 1. Get the current Q-value for the state-action pair
    current_q = Q[normalize_state(state)][action]
    
    # 2. Find the maximum Q-value for the next state
    # If the game is over, there are no legal actions, so the max future value is 0
    next_actions = legal_actions(next_state)
    if not next_actions:
        max_future_q = 0.0
    else:
        max_future_q = max(Q[normalize_state(next_state)][a] for a in next_actions)
        
    # 3. Apply the mathematical Q-learning formula
    new_q = current_q + alpha * (reward + gamma * max_future_q - current_q)
    
    # 4. Update the table
    Q[normalize_state(state)][action] = new_q

## Part C – Epsilon-Greedy Action Selection

Implement an epsilon-greedy policy.

- With probability epsilon: choose a random legal action
- Otherwise: choose the action with the highest Q-value

In [11]:
def choose_action_epsilon_greedy(Q, state, epsilon=0.1):
    """
    Choose an action using epsilon-greedy exploration.
    """
    actions = legal_actions(state)
    
    if not actions:
        return None
        
    # Explore: Try something random
    if random.random() < epsilon:
        return random.choice(actions)
        
    # Exploit: Choose the best action from the Q-table
    # If multiple actions have the same Q-value, this picks the first one it sees.
    # To be perfectly fair, we can use a small trick to break ties randomly, 
    # but using the simple max() is completely acceptable here.
    return max(actions, key=lambda a: Q[normalize_state(state)][a])

## Part D – Training by Self-Play (10 pts)

Train your agent by self-play for at least:

```python
3000 episodes
```

You may train one shared Q-table for both players.

In [10]:
def train_q_learning(n_episodes=3000, start_states=None, alpha=0.5, gamma=0.9, epsilon=0.2):
    """
    Train a Q-learning agent through self-play.
    """
    if start_states is None:
        start_states = [[1, 3, 5], [3, 5, 7], [1, 1, 2], [2, 4, 6]]

    Q = defaultdict(lambda: defaultdict(float))

    for episode in range(n_episodes):
        # Pick a random starting state and make a copy so we don't modify the original
        state = list(random.choice(start_states))
        
        while True:
            # 1. Current player chooses an action
            action = choose_action_epsilon_greedy(Q, state, epsilon)
            
            # 2. Apply the action to get the next state
            next_state = apply_action(state, action)
            
            # 3. Check if the game is over and determine the reward
            if is_terminal(next_state):
                # In Misère Nim, if your move empties the board, YOU lose.
                reward = -1.0
                update_q(Q, state, action, reward, next_state, alpha, gamma)
                break # Game is over
            else:
                # The game is not over. Standard non-terminal moves have 0 reward.
                reward = 0.0
                update_q(Q, state, action, reward, next_state, alpha, gamma)
                
                # Move to the next state for the next turn
                state = next_state

    return Q

## Part E – Learned Policy (5 pts)

After training, report the best learned move for:

```python
[1, 3, 5]
[3, 5, 7]
[1, 1, 2]
```

In [14]:
# Train the agent
Q = train_q_learning(n_episodes=5000) # Increased episodes slightly for better stability

test_states = [
    [1, 3, 5],
    [3, 5, 7],
    [1, 1, 2]
]

print("Best Learned Moves:")
for state in test_states:
    best_move = max(legal_actions(state), key=lambda a: Q[normalize_state(state)][a])
    print(f"State {state} -> Best Move: Heap {best_move[0]}, Remove {best_move[1]} objects")

Best Learned Moves:
State [1, 3, 5] -> Best Move: Heap 0, Remove 1 objects
State [3, 5, 7] -> Best Move: Heap 0, Remove 1 objects
State [1, 1, 2] -> Best Move: Heap 0, Remove 1 objects


# Layer 4 – Experimental Comparison (15 Points)

Run tournaments of 20 games each.

## Required Matchups

1. Random vs Random
2. Monte Carlo vs Random
3. Q-learning vs Random
4. Q-learning vs Monte Carlo
5. Minimax from Homework 6 vs Q-learning

If you do not reuse your Minimax agent from Homework 6, clearly state that and compare the first four matchups.

In [1]:
# --- AGENT DEFINITIONS ---

def random_agent(state):
    """
    Choose a random legal action.
    """
    return random.choice(legal_actions(state))


def mc_agent(state):
    """
    Choose an action using Monte Carlo rollouts.
    """
    action, _ = best_move_mc(state, n_trials=100)
    return action


def q_learning_agent(Q, state):
    """
    Choose the best action according to learned Q-values.
    """
    actions = legal_actions(state)
    if not actions:
        return None
    return max(actions, key=lambda a: Q[normalize_state(state)][a])


# --- TOURNAMENT LOGIC ---

def play_game(agent0, agent1, start_state):
    """
    Play one game between two agents.
    Returns: winner (0 or 1)
    """
    state = list(start_state)
    current_player = 0
    agents = [agent0, agent1]
    
    while not is_terminal(state):
        # Get the action from the current player's function
        action = agents[current_player](state)
        state = apply_action(state, action)
        
        # If the move ended the game, the current player loses in Misère Nim
        if is_terminal(state):
            return 1 - current_player # The OTHER player wins
            
        # Switch turns
        current_player = 1 - current_player


def run_tournament(agent0, agent1, start_state=[3, 5, 7], n_games=20):
    """
    Run a tournament and return win counts.
    """
    wins = {0: 0, 1: 0}
    
    for i in range(n_games):
        # Alternate who goes first to be fair
        if i % 2 == 0:
            winner = play_game(agent0, agent1, start_state)
            wins[winner] += 1
        else:
            # agent1 goes first (player 0 from the game's perspective)
            winner = play_game(agent1, agent0, start_state)
            # If player 0 won this reversed game, it means agent1 won
            if winner == 0:
                wins[1] += 1
            else:
                wins[0] += 1
                
    return wins[0], wins[1]

## Report Table 
## NOTE: Not using Minimax

Fill in a table like this:

| Matchup | Agent 1 Wins | Agent 2 Wins | Winner |
|---|---:|---:|---|
| Random vs Random | | | |
| Monte Carlo vs Random | | | |
| Q-learning vs Random | | | |
| Q-learning vs Monte Carlo | | | |
| Minimax vs Q-learning | | | |

In [18]:
# TODO: run tournaments and display results
from IPython.display import display
import pandas as pd

# 1. Create a "wrapper" for the Q-learning agent
# The run_tournament function expects agents that only take 'state' as an argument.
# Our Q-learning agent requires both the Q-table AND the state.
# We use a lambda function to "bake in" our trained Q-table so the tournament can use it.
q_agent = lambda state: q_learning_agent(Q, state)

# 2. Define our tournament schedule (Only the first 4 matchups!)
# Each tuple contains: (Agent 1 Name, Agent 1 Function, Agent 2 Name, Agent 2 Function)
matchups = [
    ("Random", random_agent, "Random", random_agent),
    ("Monte Carlo", mc_agent, "Random", random_agent),
    ("Q-learning", q_agent, "Random", random_agent),
    ("Q-learning", q_agent, "Monte Carlo", mc_agent)
]

# 3. Create an empty list to hold our final rows of data
results_data = []

# 4. Run the tournaments
print("Running tournaments... Please wait!")

for name1, func1, name2, func2 in matchups:
    # Play 20 games using the starting state [3, 5, 7]
    wins1, wins2 = run_tournament(func1, func2, start_state=[3, 5, 7], n_games=20)
    
    # Figure out who won the most games to declare an overall winner
    if wins1 > wins2:
        overall_winner = name1
    elif wins2 > wins1:
        overall_winner = name2
    else:
        overall_winner = "Tie"
        
    # Format the matchup name nicely for the left column of the table
    matchup_title = f"{name1} vs {name2}"
    
    # Add this completed row of results to our list
    results_data.append([matchup_title, wins1, wins2, overall_winner])

# 5. Display the results in a table using Pandas
# This will render a clean grid in your Jupyter Notebook that perfectly matches the requested format
df = pd.DataFrame(results_data, columns=["Matchup", "Agent 1 Wins", "Agent 2 Wins", "Winner"])

# Display the dataframe
display(df)

Running tournaments... Please wait!


,Matchup,Agent 1 Wins,Agent 2 Wins,Winner
0,Random vs Random,7,13,Random
1,Monte Carlo vs Random,20,0,Monte Carlo
2,Q-learning vs Random,7,13,Random
3,Q-learning vs Monte Carlo,2,18,Monte Carlo


## Analysis Questions

### Q1 (5 pts)

Which agent performed best overall?

**Your answer:**
Monde Carlo performed best overall. It uses real-time simulation so everytime it was its turn, it played out 100 ranodm versions of the game for every possible move, so it was basically perfect for this game.

### Q2 (5 pts)

Which method required the most computation?

**Your answer:**
Q-Learning requires the most computation because of its training. It plays thousands of self-play episodes, and that takes the most computation, but actually playing takes very little. Monte Carlo would be the most computation during gameplay because it does all of its simulations on the fly before making decisions, so both are kind of tied.

### Q3 (5 pts)

When is search better than learning?

When is learning better than search?

**Your answer:**

Search is better when the game rules are firm and  the environment is deterministic. The more stable the environment and set of actions, the more efficient search functions can do their job. Simpler games like tic-tac-toe work best with search.

Learning is better when the state space is massive, and the environment is prone to changing wildly with each passing turn. Games like Chess often would make search functions work too much and cause massive efficiency issues because there is simply too much changing for the search to keep up with.

# Code Quality Expectations

Your code should include:

- Meaningful variable names
- Clear functions
- Comments where needed
- Organized outputs
- Tables or printed summaries for experiments

# Summary

Modify rewards to encourage shorter wins or longer survival.

Did the learned policy change?

Explain.

**Your answer:**

Yes, this would change everything. Non-terminal moves reward 0.0, meaning the agent only wants to win, not caring about in how many turns it does it in. We can apply a small negative reward for non-terminal moves to create a sort of penalty. With this, the agent will now try to get its wins in smaller amounts of moves to avoid losing points. If we put the agent in mathematically-impossible-to-win situations, we can add a positive reward for every step survived, which would lead to it going for pure lasting survival.

# Hints

- Reuse Homework 6 Nim code if helpful
- Normalize states as tuples for dictionary keys
- Use the `random` module carefully
- Begin with small states before `[3, 5, 7]`
- Test helper functions before building agents

# Academic Integrity

You may discuss ideas, but all code and written work must be your own.

You may not submit another student’s implementation or an AI-generated solution without understanding and modifying it yourself.

# Final Reflection

This homework shows a major transition in AI:

- Search computes strong moves
- Simulation estimates strong moves
- Reinforcement learning discovers strong moves

That is one of the most important ideas in modern AI.